# Analysis

In [1]:
import sys
sys.path.append("..")

import numpy as np
from collections import Counter
from project1.data import load_data, EMOTIONS

dataset, outcome_space = load_data(test_size=0.2, seed=0)

### Dataset analysis

In [2]:
labels = dataset["train"]["label"]       # one integer per text
counts = Counter(labels)                 # how often each label occurs
total = len(labels)

print("Dataset sizes:", {split: len(dataset[split]) for split in dataset})
print()

print("Class distribution (train):")
for label_id in range(len(EMOTIONS)):
    n = counts[label_id]
    print(f"  {outcome_space.name(label_id):8s} {n:5d}  {100 * n / total:5.1f}%")

print()
print(f"Ratio largest to smallest class: {max(counts.values()) / min(counts.values()):.2f}")
print(f"Always predict the largest class: {max(counts.values()) / total:.3f}")
print(f"Random guessing (1/{len(EMOTIONS)}):           {1 / len(EMOTIONS):.3f}")

lengths = [len(t.split()) for t in dataset["train"]["text"]]
print()
print(f"Words per text: mean {np.mean(lengths):.1f} | median {np.median(lengths):.0f} "
      f"| min {min(lengths)} | max {max(lengths)}")

Dataset sizes: {'train': 6012, 'test': 1504}

Class distribution (train):
  joy        877   14.6%
  fear       868   14.4%
  anger      848   14.1%
  sadness    858   14.3%
  disgust    845   14.1%
  shame      863   14.4%
  guilt      853   14.2%

Ratio largest to smallest class: 1.04
Always predict the largest class: 0.146
Random guessing (1/7):           0.143

Words per text: mean 22.2 | median 19 | min 1 | max 178


### Metric: accuracy

Why accuracy is a suitable metric for this dataset:

1. **The dataset is balanced.** Accuracy can be misleading when one class dominates: a model
   that always predicts that class can score high without learning anything. In ISEAR the
   seven classes are almost equally large (largest/smallest class ratio ≈ 1.04). Always predicting
   the largest class gives about 14.6%, which is practically the same as random guessing
   (1/7 ≈ 14.3%). So a score clearly above that level reflects real learning beyond the
   class distribution.

2. **All classes matter equally.** The task is to recognise all seven emotions, and nothing
   in the task makes one emotion more important than another. Every misclassification is
   therefore equally costly. Accuracy (the fraction of correct predictions) fits this: it
   counts every error as one error, whichever two emotions were confused.

### Test set: final evaluation of the chosen system

All comparisons above used cross-validation on the training split. The test split (20%, 1504 texts) was used only here, once, for the chosen system (MLP + indicator).

We load the 15 saved models of the final run (3 seeds × 5 folds, `results/mlp_indicator/`)

In [3]:
import part1
from part1 import smlp
from project1 import encoders

SEEDS, N_FOLDS = (0, 1, 2), 5
MODEL_DIR = "../results/mlp_indicator"


def accuracy(model, encoder, texts, labels):
    probs = smlp.predict_probs(texts, model, encoder)
    return (probs.argmax(axis=1) == np.array(labels)).mean()


train_texts, train_labels = dataset["train"]["text"], dataset["train"]["label"]
test_texts, test_labels = dataset["test"]["text"], dataset["test"]["label"]

tokenizer = part1.experiment.train_tokenizer(train_texts, vocab_size=5000, min_frequency=2)
encoder = encoders.TokenIndicator(tokenizer)

val_accs, test_accs = [], []
for seed in SEEDS:
    # same CV split as in the final run, so each model is checked on its own validation fold
    folds = part1.experiment.cross_validation_split(list(zip(train_texts, train_labels)), k=N_FOLDS, seed=seed)
    for fold, split in folds.items():
        model = smlp.load(f"{MODEL_DIR}/seed-{seed}/fold-{fold}/model")
        val_texts, val_labels = zip(*split["validation"])
        val_accs.append(accuracy(model, encoder, val_texts, val_labels))
        test_accs.append(accuracy(model, encoder, test_texts, test_labels))

print(f"Check, mean validation accuracy: {np.mean(val_accs):.4f} (final run: 0.5548)")
print(f"Test accuracy MLP + indicator:   {np.mean(test_accs):.4f} ± {np.std(test_accs):.4f} "
      f"(min {min(test_accs):.4f}, max {max(test_accs):.4f}, 15 models, {len(test_labels)} texts)")

Check, mean validation accuracy: 0.5548 (final run: 0.5548)
Test accuracy MLP + indicator:   0.5504 ± 0.0064 (min 0.5379, max 0.5658, 15 models, 1504 texts)
